In [ ]:
import os
import torch
import warnings
import pdfplumber
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from transformers import pipeline
from docx import Document as DocxDocument

# Ocultar avisos chatos do terminal
warnings.filterwarnings("ignore")

# 1. Detetar a Placa Gráfica e Configurar o Dispositivo 
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Motor de Processamento: {dispositivo.upper()}")


# --- FUNÇÃO PERSONALIZADA: LEITOR INTELIGENTE (PDF e DOCX) ---
def ler_documentos_inteligente(pasta):
    documentos_langchain = []
    
    for ficheiro in os.listdir(pasta):
        caminho_ficheiro = os.path.join(pasta, ficheiro)
        
        # 1. LER FICHEIROS WORD (.docx) - Ex: Listas de Verificação
        if ficheiro.lower().endswith(".docx"):
            try:
                doc_word = DocxDocument(caminho_ficheiro)
                texto = "\n".join([paragrafo.text for paragrafo in doc_word.paragraphs if paragrafo.text.strip()])
                
                if texto.strip():
                    doc = Document(page_content=texto, metadata={"source": caminho_ficheiro, "tipo": "Checklist/Word"})
                    documentos_langchain.append(doc)
            except Exception as e:
                print(f"Erro ao ler o ficheiro Word {ficheiro}: {e}")

        # 2. LER FICHEIROS PDF (.pdf) - Ex: DRs e Currículos
        elif ficheiro.lower().endswith(".pdf"):
            with pdfplumber.open(caminho_ficheiro) as pdf:
                # Tirar uma "amostra" da primeira página para detetar o tipo de PDF
                amostra = pdf.pages[0].extract_text() or ""
                is_diario_republica = "Diário da República" in amostra
                
                for num_pagina, page in enumerate(pdf.pages):
                    texto_completo = ""
                    
                    if is_diario_republica:
                        # Recortar em 2 Colunas (Ler cada metade da página)
                        largura = page.width
                        altura = page.height
                        meio = largura / 2
                        
                        texto_esq = page.within_bbox((0, 0, meio, altura)).extract_text() or ""
                        texto_dir = page.within_bbox((meio, 0, largura, altura)).extract_text() or ""
                        texto_completo = texto_esq + "\n\n" + texto_dir
                    else:
                        # Modo PDF Normal (Ler a página inteira)
                        texto_completo = page.extract_text() or ""
                    
                    if texto_completo.strip():
                        doc = Document(
                            page_content=texto_completo, 
                            metadata={"source": caminho_ficheiro, "page": num_pagina + 1, "tipo": "Diário da República" if is_diario_republica else "PDF Normal"}
                        )
                        documentos_langchain.append(doc)
                        
    return documentos_langchain
# ----------------------------------------------------------------------

# 2. Carregar os PDFs
pasta_pdfs = "../pdfs_teste"
if not os.path.exists(pasta_pdfs):
    os.makedirs(pasta_pdfs)
    print(f"Criei a pasta '{pasta_pdfs}'. Por favor, coloca lá um PDF e volta a correr a célula!")
else:
    documentos = ler_documentos_inteligente(pasta_pdfs)
    
    if len(documentos) == 0:
        print(f"A pasta '{pasta_pdfs}' está vazia ou os PDFs não têm texto. Coloca lá um PDF para testarmos!")
    else:
        print(f"Sucesso: Foram carregadas {len(documentos)} páginas de PDF em formato de coluna dupla.")

        # 3. Partir o texto do PDF em blocos mais pequenos
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200)
        textos_partidos = text_splitter.split_documents(documentos)

        # 4. Criar a Base de Dados Vetorial 
        print("A construir a base de conhecimento vetorial...")
        embeddings = HuggingFaceEmbeddings(
            model_name="all-MiniLM-L6-v2",
            model_kwargs={'device': dispositivo}
        )
        base_dados_vetorial = FAISS.from_documents(textos_partidos, embeddings)
        retriever = base_dados_vetorial.as_retriever(search_kwargs={"k": 6})

        # 5. Carregar o Qwen2.5 3B 
        model_id = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit" 
        
        print("A carregar o Qwen2.5 3B")

        gerador_texto = pipeline(
            "text-generation",
            model=model_id,
            model_kwargs={
                "torch_dtype": torch.float16,
                "low_cpu_mem_usage": True
            },
            max_new_tokens=256,
            temperature=0.1, 
            do_sample=True,
            repetition_penalty=1.1,
            device_map="auto" 
        )
        llm_local = HuggingFacePipeline(pipeline=gerador_texto)


        # 6. A TUA PERGUNTA AO PDF
        pergunta = input("Escreve a tua pergunta sobre o conteúdo do PDF: ")
        
        # Pesquisar na base de dados apenas UMA VEZ
        documentos_relevantes = retriever.invoke(pergunta)
        contexto = "\n".join([doc.page_content for doc in documentos_relevantes])

        # Mostrar o que o sistema "leu" (Debug)
        print(f"\n\n --- O que o sistema encontrou no PDF (Contexto) ---")
        print(contexto)
        print("------------------------------------------------------\n\n")

        # 7. Prompt Anti-Alucinação e Cópia Exata
        prompt = f"""<|im_start|>system
És um assistente de extração de dados estritamente factual.

REGRAS ANTI-ALUCINAÇÃO E RIGOR JURÍDICO:
1. BASE FATUAL EXCLUSIVA: Responde APENAS com base no contexto fornecido. Se a resposta à pergunta NÃO ESTIVER no texto, escreve OBRIGATORIAMENTE "Informação não encontrada no contexto". Nunca inventes leis, datas ou coimas.
2. PRECISÃO LEGAL: Mantém a redação exata de termos jurídicos, números de leis, decretos, portarias e artigos. Não tentes reescrever a lei por tuas palavras se isso alterar o seu sentido.
3. CITAÇÃO DA FONTE: Sempre que o contexto indicar a origem da regra, inicia ou termina a tua resposta citando-a (ex: "Segundo o Artigo 4.º...", ou "Conforme a Portaria X...").
4. RESPOSTA DIRETA: RESPOSTA DIRETA: Responde apenas ao que foi perguntado de forma o mais curta possível, sem texto de "enchimento" ou introduções simpáticas.
5. ZERO RACIOCÍNIO: Não expliques como chegaste à resposta, não faças suposições nem descrevas os artigos. Dá a resposta final de forma direta e limpa.<|im_end|>
<|im_start|>user
CONTEXTO:
{contexto}

PERGUNTA: {pergunta}<|im_end|>
<|im_start|>assistant
"""
        print(f"Pergunta: {pergunta}")
        resposta = llm_local.invoke(prompt)
        
        # Limpeza para Qwen
        resposta_limpa = resposta.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()
        print(f"\n--- RESULTADO DA EXTRAÇÃO ---\n{resposta_limpa}")